# Módulo 0 — Análisis climático histórico: Jocolí, Lavalle, Mendoza

Este módulo presenta el análisis de los datos climáticos históricos del período 1990–2024 para la localidad de Jocolí, departamento de Lavalle, provincia de Mendoza. Los datos se obtienen de la API de reanálisis ERA5-Land de Open-Meteo, con resolución temporal horaria y resolución espacial de aproximadamente 9 km.

El objetivo es caracterizar el régimen térmico e hídrico del sitio y evaluar su aptitud para el cultivo de pistacho (*Pistacia vera* L.), con especial atención a las variables que condicionan el rendimiento: acumulación de frío invernal, calor estival y déficit hídrico.

**Variables analizadas:** temperatura del aire a 2 m · precipitación · evapotranspiración de referencia (ET₀, método FAO Penman-Monteith)  
**Período:** 1990–2024 (35 años)  
**Fuente:** Open-Meteo Archive API — ERA5-Land (ECMWF)

In [1]:
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO_URL = "https://github.com/Emilialandgrebe/tesis-maestria.git"
    ROOT = Path("/content/tesis-maestria")
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt"), "-q"],
        check=True,
    )
else:
    ROOT = Path().resolve().parent  # notebooks/ -> raiz del proyecto

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Entorno : {'Google Colab' if EN_COLAB else 'local'}")
print(f"ROOT    : {ROOT}")

Entorno : Google Colab
ROOT    : /content/tesis-maestria


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymannkendall as mk
from scipy import stats
from scipy.stats import theilslopes
from statsmodels.nonparametric.smoothers_lowess import lowess

from src.data.climate_fetcher import fetch_climate_data
from src.data.climate_features import (
    calcular_calor_verano,
    calcular_deficit_hidrico,
    calcular_heladas_tardias,
    calcular_horas_frio,
)

np.random.seed(42)

df = fetch_climate_data()

horas_frio      = calcular_horas_frio(df)
heladas         = calcular_heladas_tardias(df)
calor_verano    = calcular_calor_verano(df)
deficit_hidrico = calcular_deficit_hidrico(df)

In [ ]:
print("=" * 60)
print("RANGO DE FECHAS")
print("=" * 60)
print(f"Inicio : {df.index.min()}")
print(f"Fin    : {df.index.max()}")
print(f"Total  : {len(df):,} registros horarios")
print(f"         ({len(df) / 8760:.1f} años aprox.)")

In [ ]:
print("PRIMERAS FILAS")
df.head()

In [ ]:
print("ÚLTIMAS FILAS")
df.tail()

In [ ]:
print("ESTRUCTURA DEL DATAFRAME")
df.info()

In [ ]:
print("ESTADÍSTICAS DESCRIPTIVAS")
df.describe()

In [ ]:
print("=" * 60)
print("VALORES NULOS POR COLUMNA")
print("=" * 60)
nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(3)
resumen_nulos = nulos.to_frame("cantidad").assign(porcentaje=pct)
print(resumen_nulos)
print()
if nulos.sum() == 0:
    print("Sin valores nulos. Dataset completo.")
else:
    print(f"ATENCIÓN: {nulos.sum()} valores nulos en total.")

## Temperatura máxima diaria media en verano (enero–febrero)

Se calcula la temperatura máxima diaria media durante los meses de enero y febrero de cada año del período analizado. Estos meses corresponden a la etapa de mayor demanda térmica del pistacho, cuando el cultivo requiere temperaturas elevadas para completar el llenado y la apertura de cáscara del fruto. El rango óptimo para esta etapa es de 35 a 38 °C (Crane y Takeda, 1979; Ferguson, 2006).

Para el análisis de tendencia se aplica el **test de Mann-Kendall modificado** (Hamed y Rao, 1998) y la magnitud de la tendencia se estima mediante el **estimador de Sen**. La incertidumbre de la pendiente se representa como el intervalo de confianza del 95% del estimador, que constituye una banda de variación estadísticamente válida. Se incorpora además un suavizado LOWESS (fracción 0,25) para capturar la estructura no lineal de la variabilidad interanual.

Los años extremos individuales y la distribución por décadas se presentan en secciones específicas, con el fin de no sobrecargar esta figura con información que corresponde a análisis distintos.

In [ ]:
años = calor_verano.index.values.astype(float)
temperaturas = calor_verano.values

mk_result = mk.hamed_rao_modification_test(temperaturas)
sen = theilslopes(temperaturas, años)

linea_sen      = sen.intercept + sen.slope * años
linea_sen_low  = sen.intercept + sen.low_slope * años
linea_sen_high = sen.intercept + sen.high_slope * años

suavizado = lowess(temperaturas, años, frac=0.25)

C_SERIE     = "#2C5F8A"
C_TENDENCIA = "#404040"
C_LOWESS    = "#1A1A1A"
C_OPTIMO    = "#4A6741"

fig, ax = plt.subplots(figsize=(13, 6))

# Serie principal
ax.plot(años, temperaturas, color=C_SERIE, linewidth=2,
        marker="o", markersize=4, label="Tmax media ene–feb")

# Rango óptimo del pistacho
ax.fill_between(años, 35, 38, color=C_OPTIMO, alpha=0.10,
                label="Rango óptimo pistacho (35–38 °C)")
ax.axhline(35, color=C_OPTIMO, linewidth=0.8, linestyle="--")
ax.axhline(38, color=C_OPTIMO, linewidth=0.8, linestyle="--")

# Sen's Slope + IC 95% como banda sombreada
ax.fill_between(años, linea_sen_low, linea_sen_high,
                color=C_TENDENCIA, alpha=0.10, label="IC 95% Sen's Slope")
ax.plot(años, linea_sen, color=C_TENDENCIA, linestyle="--", linewidth=1.8,
        label=(f"Sen's Slope = {sen.slope:.3f} °C/año\n"
               f"Mann-Kendall τ = {mk_result.Tau:.2f}   "
               f"p = {mk_result.p:.3f}   ({mk_result.trend})"))

# Suavizado LOWESS
ax.plot(suavizado[:, 0], suavizado[:, 1], color=C_LOWESS,
        linewidth=2, label="Suavizado LOWESS (frac = 0,25)")

ax.set_title(
    "Temperatura máxima diaria media del verano (enero–febrero)\n"
    "Jocolí, Lavalle, Mendoza (1990–2024)",
    fontsize=13, fontweight="bold", loc="center",
)
ax.set_xlabel("Año", fontsize=11)
ax.set_ylabel("Temperatura (°C)", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.4, color="#AAAAAA")
ax.legend(loc="upper left", fontsize=9.5, framealpha=0.9)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia a partir de datos ERA5-Land (Open-Meteo).",
            ha="right", fontsize=8, color="#555555")
plt.show()

print("─" * 52)
print("RESULTADOS DEL ANÁLISIS DE TENDENCIA")
print("─" * 52)
print(f"  Tendencia detectada : {mk_result.trend}")
print(f"  Kendall's τ         : {mk_result.Tau:.4f}")
print(f"  p-valor             : {mk_result.p:.4f}")
print(f"  Estadístico z       : {mk_result.z:.4f}")
print()
print(f"  Sen's Slope         : {sen.slope:.4f} °C/año")
print(f"  IC 95%              : [{sen.low_slope:.4f}, {sen.high_slope:.4f}]")


## Frecuencia de días con temperatura extrema (enero–febrero)

Se cuantifica el número de días por año en que la temperatura máxima diaria supera los umbrales de 35 °C, 38 °C y 40 °C durante el período estival (enero–febrero). Este indicador resulta agronómicamente más informativo que la temperatura media, ya que permite evaluar la intensidad y acumulación del estrés térmico al que se expone el cultivo durante la etapa crítica de maduración.

Valores superiores a 38 °C marcan el límite del rango óptimo para el pistacho, mientras que temperaturas por encima de los 40 °C se asocian a daño directo en el fruto, reducción del calibre comercial y mayor incidencia de frutos vacíos (blank nuts).

In [ ]:
mascara_verano = df.index.month.isin([1, 2])
tmax_diaria_verano = (
    df.loc[mascara_verano, "temperature_2m"]
    .groupby(df.index[mascara_verano].date)
    .max()
)
tmax_diaria_verano.index = pd.to_datetime(tmax_diaria_verano.index)

dias35 = tmax_diaria_verano.groupby(tmax_diaria_verano.index.year).apply(lambda x: (x > 35).sum())
dias38 = tmax_diaria_verano.groupby(tmax_diaria_verano.index.year).apply(lambda x: (x > 38).sum())
dias40 = tmax_diaria_verano.groupby(tmax_diaria_verano.index.year).apply(lambda x: (x > 40).sum())

# Paleta sobria: gradiente cálido según intensidad del umbral
C_35 = "#D9A86C"
C_38 = "#B05C2E"
C_40 = "#7A1C1C"

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(dias35.index, dias35.values, color=C_35, linewidth=2.2,
        marker="o", markersize=4, label=">35 °C")
ax.plot(dias38.index, dias38.values, color=C_38, linewidth=2.2,
        marker="o", markersize=4, label=">38 °C")
ax.plot(dias40.index, dias40.values, color=C_40, linewidth=2.2,
        marker="o", markersize=4, label=">40 °C")

ax.set_title(
    "Frecuencia anual de días con temperatura extrema (enero–febrero)\n"
    "Jocolí, Lavalle, Mendoza (1990–2024)",
    fontsize=13, fontweight="bold", loc="center",
)
ax.set_ylabel("Número de días", fontsize=11)
ax.set_xlabel("Año", fontsize=11)
ax.grid(alpha=0.3, linestyle=":", color="#AAAAAA")
ax.legend(fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia a partir de datos ERA5-Land (Open-Meteo).",
            ha="right", fontsize=8, color="#555555")
plt.show()


## Distribución de la temperatura máxima diaria por década

Se analiza la distribución estadística de las temperaturas máximas diarias de enero–febrero agrupadas por décadas. Este enfoque permite visualizar no solo el desplazamiento de la mediana a lo largo del tiempo, sino también los cambios en la variabilidad interna y en los valores extremos, información que la temperatura media anual por sí sola no refleja.

Cada caja muestra el rango intercuartílico (P25–P75), la línea central indica la mediana y los puntos exteriores corresponden a valores atípicos según el criterio de 1,5 × IQR. Las líneas horizontales de referencia señalan los límites del rango óptimo de temperatura para el cultivo (35 y 38 °C).

In [ ]:
decadas_etiquetas = {
    1990: "1990–1999",
    2000: "2000–2009",
    2010: "2010–2019",
    2020: "2020–2024",
}
# Gradiente de azul frío a rojo cálido por década
colores_decadas = {
    1990: "#7BA7C0",
    2000: "#4E7FA8",
    2010: "#B87448",
    2020: "#8B2020",
}

decada_serie = (tmax_diaria_verano.index.year // 10) * 10

datos_boxplot = [
    tmax_diaria_verano[decada_serie == d].values
    for d in sorted(decadas_etiquetas)
]
etiquetas     = [decadas_etiquetas[d] for d in sorted(decadas_etiquetas)]
colores_lista = [colores_decadas[d]   for d in sorted(decadas_etiquetas)]

fig, ax = plt.subplots(figsize=(10, 6))

bp = ax.boxplot(
    datos_boxplot,
    labels=etiquetas,
    patch_artist=True,
    widths=0.5,
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(linewidth=1.4, color="#444444"),
    capprops=dict(linewidth=1.4, color="#444444"),
    flierprops=dict(marker="o", markersize=3.5, linestyle="none",
                    alpha=0.5, markerfacecolor="#666666", markeredgecolor="none"),
)
for patch, color in zip(bp["boxes"], colores_lista):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

C_OPTIMO = "#4A6741"
ax.axhline(35, color=C_OPTIMO, linewidth=1, linestyle="--",
           label="35 °C (límite inferior rango óptimo)")
ax.axhline(38, color=C_OPTIMO, linewidth=1, linestyle="-.",
           label="38 °C (límite superior rango óptimo)")

ax.set_title(
    "Distribución de la temperatura máxima diaria en verano (ene–feb) por década\n"
    "Jocolí, Lavalle, Mendoza (1990–2024)",
    fontsize=13, fontweight="bold", loc="center",
)
ax.set_ylabel("Temperatura máxima diaria (°C)", fontsize=11)
ax.set_xlabel("Década", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.4, color="#AAAAAA")
ax.legend(fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia a partir de datos ERA5-Land (Open-Meteo).",
            ha="right", fontsize=8, color="#555555")
plt.show()


## Anomalías térmicas respecto al período de referencia 1990–2020

Se calculan las anomalías de temperatura máxima media de verano tomando como período de referencia los años 1990–2020, criterio estándar establecido por la Organización Meteorológica Mundial (OMM) para la caracterización climatológica. La anomalía de cada año se obtiene como la diferencia entre el valor observado y la media del período de referencia.

Las barras positivas indican años más cálidos que el promedio histórico y las barras negativas corresponden a años más frescos. Se ajusta además una tendencia lineal sobre las anomalías para cuantificar el ritmo de calentamiento en el período analizado, expresado en °C por año.

In [ ]:
ref_media   = calor_verano.loc[1990:2020].mean()
anomalias   = calor_verano - ref_media
colores_bar = ["#B5432A" if v >= 0 else "#4472C4" for v in anomalias]

# Mann-Kendall y Sen's Slope sobre las anomalías
mk_an  = mk.hamed_rao_modification_test(anomalias.values)
sen_an = theilslopes(anomalias.values, anomalias.index.astype(float))
linea_sen_an = sen_an.intercept + sen_an.slope * anomalias.index.astype(float)

fig, ax = plt.subplots(figsize=(13, 5))

ax.bar(anomalias.index, anomalias.values, color=colores_bar, width=0.8, alpha=0.85)
ax.axhline(0, color="#222222", linewidth=0.9)

ax.plot(
    anomalias.index, linea_sen_an,
    color="#404040", linestyle="--", linewidth=1.8,
    label=(f"Sen's Slope = {sen_an.slope:.3f} °C/año  "
           f"| MK τ={mk_an.Tau:.2f}   p={mk_an.p:.3f}   {mk_an.trend}"),
)

ax.set_title(
    "Anomalías de temperatura máxima media en verano respecto al período base 1990–2020\n"
    "Jocolí, Lavalle, Mendoza",
    fontsize=13, fontweight="bold", loc="center",
)
ax.set_xlabel("Año", fontsize=11)
ax.set_ylabel("Anomalía (°C)", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.4, color="#AAAAAA")
ax.legend(fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia a partir de datos ERA5-Land (Open-Meteo).",
            ha="right", fontsize=8, color="#555555")
plt.show()

print(f"\nMedia de referencia 1990–2020 : {ref_media:.2f} °C")
print(f"Anomalía promedio 2021–2024   : {anomalias.loc[2021:].mean():+.2f} °C")


## Grados-Día de Crecimiento en el período de brotación (agosto–octubre)

Se calculan los Grados-Día de Crecimiento (GDD) acumulados durante los meses de agosto, septiembre y octubre, que corresponden al período de brotación y floración del pistacho en el hemisferio sur. Los GDD se obtienen como la suma diaria de la diferencia entre la temperatura media del aire y la temperatura base de 10 °C, descartando los días en que esa diferencia resulta negativa.

Este indicador complementa el análisis de temperatura máxima media, ya que cuantifica el calor disponible para el desarrollo fenológico del cultivo en la etapa que precede a la maduración del fruto. Un incremento sostenido en los GDD de primavera puede implicar un adelantamiento de la floración y una mayor asincronía entre la floración masculina y femenina, lo que afecta directamente la polinización cruzada y el cuajado de frutos en una especie dioica como el pistacho.

In [ ]:
BASE_TEMP_GDD = 10.0  # temperatura base estándar para pistacho (°C)

mascara_brotacion = df.index.month.isin([8, 9, 10])
tmed_diaria_brotacion = (
    df.loc[mascara_brotacion, "temperature_2m"]
    .groupby(df.index[mascara_brotacion].date)
    .mean()
)
tmed_diaria_brotacion.index = pd.to_datetime(tmed_diaria_brotacion.index)

gdd_diario = (tmed_diaria_brotacion - BASE_TEMP_GDD).clip(lower=0)
gdd_anual  = (
    gdd_diario.groupby(gdd_diario.index.year)
    .sum()
    .rename("gdd_brotacion")
    .round(1)
)

mk_gdd   = mk.hamed_rao_modification_test(gdd_anual.values)
sen_gdd  = theilslopes(gdd_anual.values, gdd_anual.index.astype(float))
linea_gdd      = sen_gdd.intercept + sen_gdd.slope      * gdd_anual.index.astype(float)
linea_gdd_low  = sen_gdd.intercept + sen_gdd.low_slope  * gdd_anual.index.astype(float)
linea_gdd_high = sen_gdd.intercept + sen_gdd.high_slope * gdd_anual.index.astype(float)

fig, ax = plt.subplots(figsize=(13, 5))

ax.bar(gdd_anual.index, gdd_anual.values, color="#5C7FA3", alpha=0.75,
       label=f"GDD acumulados (base {BASE_TEMP_GDD:.0f} °C)")

ax.fill_between(gdd_anual.index, linea_gdd_low, linea_gdd_high,
                color="#404040", alpha=0.10, label="IC 95% Sen's Slope")
ax.plot(gdd_anual.index, linea_gdd, color="#404040", linewidth=1.8, linestyle="--",
        label=(f"Sen's Slope = {sen_gdd.slope:.1f} GDD/año  "
               f"| MK τ={mk_gdd.Tau:.2f}   p={mk_gdd.p:.3f}   {mk_gdd.trend}"))

ax.set_title(
    f"Grados-Día de Crecimiento en el período de brotación (agosto–octubre)\n"
    f"Jocolí, Lavalle, Mendoza (1990–2024) · base {BASE_TEMP_GDD:.0f} °C",
    fontsize=13, fontweight="bold", loc="center",
)
ax.set_xlabel("Año", fontsize=11)
ax.set_ylabel(f"GDD acumulados (base {BASE_TEMP_GDD:.0f} °C)", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.4, color="#AAAAAA")
ax.legend(fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia a partir de datos ERA5-Land (Open-Meteo).",
            ha="right", fontsize=8, color="#555555")
plt.show()

print("─" * 52)
print("RESULTADOS DEL ANÁLISIS DE TENDENCIA — GDD")
print("─" * 52)
print(f"  Tendencia detectada : {mk_gdd.trend}")
print(f"  Kendall's τ         : {mk_gdd.Tau:.4f}")
print(f"  p-valor             : {mk_gdd.p:.4f}")
print(f"  Sen's Slope         : {sen_gdd.slope:.2f} GDD/año")
print(f"  IC 95%              : [{sen_gdd.low_slope:.2f}, {sen_gdd.high_slope:.2f}]")
print(f"\nGDD medios 1990–2000 : {gdd_anual.loc[1990:2000].mean():.0f}")
print(f"GDD medios 2015–2024 : {gdd_anual.loc[2015:2024].mean():.0f}")


## Déficit hídrico anual (ET₀ − Precipitación)

Se estima el déficit hídrico anual como la diferencia entre la evapotranspiración de referencia (ET₀) y la precipitación total acumulada en el año, ambas expresadas en milímetros. Este indicador representa la demanda neta de agua de riego necesaria para cubrir la evapotranspiración del cultivo, bajo el supuesto de que no se considera el aporte de agua freática ni las pérdidas por eficiencia del sistema de riego.

La zona de Jocolí se localiza en el piedemonte árido mendocino, donde la precipitación media anual no supera los 150 mm, de modo que la práctica totalidad de la demanda hídrica del cultivo debe cubrirse mediante riego superficial o por goteo. Se incluye una línea de referencia en 600 mm, valor que corresponde al aporte mínimo de riego contemplado en el plan de negocios del proyecto analizado. La tendencia temporal del déficit se estima mediante el test de Mann-Kendall modificado y el estimador de Sen, con el mismo criterio metodológico aplicado al resto de las variables climáticas.

In [ ]:
años_dh    = deficit_hidrico.index.values.astype(float)
valores_dh = deficit_hidrico.values

mk_dh  = mk.hamed_rao_modification_test(valores_dh)
sen_dh = theilslopes(valores_dh, años_dh)
linea_dh = sen_dh.intercept + sen_dh.slope * años_dh

fig, ax = plt.subplots(figsize=(13, 5))

ax.bar(deficit_hidrico.index, valores_dh, color="#4472C4", alpha=0.80,
       label="Déficit hídrico anual (mm)")

ax.axhline(600, color="#8B2020", linewidth=1.8, linestyle="--",
           label="600 mm (mínimo de riego — plan de negocios)")

ax.plot(deficit_hidrico.index, linea_dh, color="#404040", linewidth=1.8,
        linestyle="-.",
        label=(f"Sen's Slope = {sen_dh.slope:.1f} mm/año  "
               f"| MK τ={mk_dh.Tau:.2f}   p={mk_dh.p:.3f}   {mk_dh.trend}"))

ax.set_title(
    "Déficit hídrico anual (ET₀ − Precipitación)\n"
    "Jocolí, Lavalle, Mendoza (1990–2024)",
    fontsize=13, fontweight="bold", loc="center",
)
ax.set_xlabel("Año", fontsize=11)
ax.set_ylabel("mm / año", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.4, color="#AAAAAA")
ax.legend(fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia a partir de datos ERA5-Land (Open-Meteo).",
            ha="right", fontsize=8, color="#555555")
plt.show()

print("─" * 52)
print("RESULTADOS DEL ANÁLISIS DE TENDENCIA — DÉFICIT HÍDRICO")
print("─" * 52)
print(f"  Tendencia detectada : {mk_dh.trend}")
print(f"  Kendall's τ         : {mk_dh.Tau:.4f}")
print(f"  p-valor             : {mk_dh.p:.4f}")
print(f"  Sen's Slope         : {sen_dh.slope:.2f} mm/año")
print(f"  IC 95%              : [{sen_dh.low_slope:.2f}, {sen_dh.high_slope:.2f}]")
print(f"\nDéficit medio 1990–2000: {deficit_hidrico.loc[1990:2000].mean():.0f} mm")
print(f"Déficit medio 2015–2024: {deficit_hidrico.loc[2015:2024].mean():.0f} mm")

## Índice de aridez de De Martonne

Se calcula el índice de aridez de De Martonne (IA), definido como el cociente entre la precipitación anual acumulada (P, en mm) y la suma de la temperatura media anual más 10: IA = P / (T + 10). Este índice es uno de los más utilizados en climatología agrícola de zonas mediterráneas y áridas para clasificar el régimen hídrico de una localidad (De Martonne, 1926).

Los umbrales de clasificación son los siguientes: IA < 5 corresponde a clima hiperárido; 5 ≤ IA < 10 a árido; 10 ≤ IA < 20 a semiárido. Se analiza si el índice muestra una tendencia hacia condiciones aún más secas en el período estudiado, lo que tendría implicaciones directas sobre la demanda de riego y la disponibilidad hídrica futura del proyecto.

In [ ]:
precip_anual = df["precipitation"].groupby(df.index.year).sum()
tmed_anual   = df["temperature_2m"].groupby(df.index.year).mean()

de_martonne  = (precip_anual / (tmed_anual + 10)).rename("de_martonne").round(2)
años_dm      = de_martonne.index.values.astype(float)

mk_dm  = mk.hamed_rao_modification_test(de_martonne.values)
sen_dm = theilslopes(de_martonne.values, años_dm)
linea_dm = sen_dm.intercept + sen_dm.slope * años_dm

fig, ax = plt.subplots(figsize=(13, 5))

ax.bar(de_martonne.index, de_martonne.values, color="#5C7FA3", alpha=0.75,
       label="Índice de De Martonne (IA)")

ax.axhline(5,  color="#8B2020", linewidth=1.2, linestyle="--",
           label="IA = 5 (límite árido / hiperárido)")
ax.axhline(10, color="#B05C2E", linewidth=1.2, linestyle="--",
           label="IA = 10 (límite semiárido / árido)")

ax.plot(de_martonne.index, linea_dm, color="#404040", linewidth=1.8, linestyle="-.",
        label=(f"Sen's Slope = {sen_dm.slope:.3f} /año  "
               f"| MK τ={mk_dm.Tau:.2f}   p={mk_dm.p:.3f}   {mk_dm.trend}"))

ax.set_title(
    "Índice de aridez de De Martonne [IA = P / (T + 10)]\n"
    "Jocolí, Lavalle, Mendoza (1990–2024)",
    fontsize=13, fontweight="bold", loc="center",
)
ax.set_xlabel("Año", fontsize=11)
ax.set_ylabel("Índice de De Martonne (IA)", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.4, color="#AAAAAA")
ax.legend(fontsize=9.5, framealpha=0.9)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia a partir de datos ERA5-Land (Open-Meteo).",
            ha="right", fontsize=8, color="#555555")
plt.show()

media_dm = de_martonne.mean()
clasif   = "hiperárido" if media_dm < 5 else ("árido" if media_dm < 10 else "semiárido")
print(f"IA medio 1990–2024 : {media_dm:.2f}  →  {clasif}")
print(f"IA medio 1990–2000 : {de_martonne.loc[1990:2000].mean():.2f}")
print(f"IA medio 2015–2024 : {de_martonne.loc[2015:2024].mean():.2f}")


## Horas de frío acumuladas (mayo–septiembre)

Se contabilizan las horas con temperatura del aire inferior a 7 °C durante el período de dormición del pistacho, comprendido entre los meses de mayo y septiembre. Este indicador es determinante para la ruptura de la dormición y la correcta brotación del cultivo en la primavera siguiente. Se considera un umbral crítico de 800 horas, por debajo del cual se espera una penalidad significativa sobre el rendimiento, y un umbral de 1 000 horas a partir del cual no se aplica ninguna reducción productiva.

Se analiza la tendencia de la serie mediante el test de Mann-Kendall modificado y el estimador de Sen, con el mismo criterio metodológico aplicado al análisis de temperatura.

In [ ]:
UMBRAL_FRIO_CRITICO  = 800
UMBRAL_FRIO_OPTIMO   = 1_000

años_hf    = horas_frio.index.values.astype(float)
valores_hf = horas_frio.values

mk_hf  = mk.hamed_rao_modification_test(valores_hf)
sen_hf = theilslopes(valores_hf, años_hf)
linea_hf = sen_hf.intercept + sen_hf.slope * años_hf

fig, ax = plt.subplots(figsize=(13, 5))

ax.bar(años_hf, valores_hf, color="#4472C4", alpha=0.78,
       label="Horas de frío acumuladas (< 7 °C)")

ax.axhline(UMBRAL_FRIO_CRITICO, color="#8B2020", linewidth=1.8, linestyle="--",
           label=f"Umbral crítico ({UMBRAL_FRIO_CRITICO} hs)")
ax.axhline(UMBRAL_FRIO_OPTIMO, color="#4A6741", linewidth=1.2, linestyle="--",
           label=f"Sin penalidad de rendimiento ({UMBRAL_FRIO_OPTIMO} hs)")

ax.plot(años_hf, linea_hf, color="#404040", linewidth=1.8, linestyle="-.",
        label=(f"Sen's Slope = {sen_hf.slope:.1f} hs/año  "
               f"| MK τ={mk_hf.Tau:.2f}   p={mk_hf.p:.3f}   {mk_hf.trend}"))

ax.set_title(
    "Horas de frío acumuladas durante el período de dormición (mayo–septiembre)\n"
    "Jocolí, Lavalle, Mendoza (1990–2024)",
    fontsize=13, fontweight="bold", loc="center",
)
ax.set_xlabel("Año", fontsize=11)
ax.set_ylabel("Horas de frío (< 7 °C)", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.4, color="#AAAAAA")
ax.legend(fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia a partir de datos ERA5-Land (Open-Meteo).",
            ha="right", fontsize=8, color="#555555")
plt.show()

print("─" * 52)
print("RESULTADOS DEL ANÁLISIS DE TENDENCIA — HORAS DE FRÍO")
print("─" * 52)
print(f"  Tendencia detectada : {mk_hf.trend}")
print(f"  Kendall's τ         : {mk_hf.Tau:.4f}")
print(f"  p-valor             : {mk_hf.p:.4f}")
print(f"  Sen's Slope         : {sen_hf.slope:.2f} hs/año")
print(f"  IC 95%              : [{sen_hf.low_slope:.2f}, {sen_hf.high_slope:.2f}]")
print(f"\nAños con horas < {UMBRAL_FRIO_CRITICO} hs (año crítico): "
      f"{(horas_frio < UMBRAL_FRIO_CRITICO).sum()} de {len(horas_frio)}")


## Validación de la distribución estadística de las horas de frío

La simulación de Monte Carlo del módulo siguiente requiere asumir una distribución de probabilidad para las horas de frío anuales. Se evalúa la normalidad de la serie mediante el **test de Shapiro-Wilk** (Shapiro y Wilk, 1965), que es el test más potente para muestras pequeñas (N < 50). Un valor p superior a 0,05 indica que no se rechaza la hipótesis de normalidad.

Se ajustan tres distribuciones candidatas — Normal, Log-normal y Gamma — y se comparan mediante el **Criterio de Información de Akaike (AIC)**, que penaliza la complejidad del modelo. La distribución con menor AIC es la que mejor equilibra el ajuste y la parsimonia. Se complementa el análisis con un **gráfico Q-Q** (*quantile-quantile*) para evaluar visualmente el ajuste de la distribución seleccionada.

In [ ]:
x = horas_frio.values

# ── Test de Shapiro-Wilk ──────────────────────────────────────────────────
sw_stat, sw_p = stats.shapiro(x)

# ── Ajuste de distribuciones candidatas ──────────────────────────────────
_candidatas = {
    "Normal":     stats.norm,
    "Log-normal": stats.lognorm,
    "Gamma":      stats.gamma,
}
_colores = {
    "Normal":     "#2C5F8A",
    "Log-normal": "#B05C2E",
    "Gamma":      "#4A6741",
}

resultados_ajuste = {}
for nombre, dist in _candidatas.items():
    params   = dist.fit(x)
    log_l    = dist.logpdf(x, *params).sum()
    aic      = 2 * len(params) - 2 * log_l
    ks_s, ks_p = stats.kstest(x, dist.cdf, args=params)
    resultados_ajuste[nombre] = {
        "dist": dist, "params": params,
        "aic": aic, "ks_stat": ks_s, "ks_p": ks_p,
    }

mejor_nombre = min(resultados_ajuste, key=lambda k: resultados_ajuste[k]["aic"])

# ── Figura: histograma + distribuciones ajustadas | Q-Q plot ─────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo — histograma + PDFs ajustadas
ax1.hist(x, bins=9, density=True, color="#AAAAAA", alpha=0.55,
         edgecolor="white", label="Datos observados")

x_grid = np.linspace(x.min() - 100, x.max() + 100, 300)
for nombre, res in resultados_ajuste.items():
    pdf = res["dist"].pdf(x_grid, *res["params"])
    estilo = "-" if nombre == mejor_nombre else "--"
    ax1.plot(x_grid, pdf, color=_colores[nombre], linewidth=2,
             linestyle=estilo,
             label=f"{nombre}  (AIC={res['aic']:.1f}, KS p={res['ks_p']:.3f})")

ax1.axvline(UMBRAL_FRIO_CRITICO, color="#8B2020", linewidth=1.5,
            linestyle=":", label=f"Umbral crítico ({UMBRAL_FRIO_CRITICO} hs)")
ax1.set_title("Ajuste de distribuciones de probabilidad\nHoras de frío (1990–2024)",
              fontsize=12, fontweight="bold", loc="center")
ax1.set_xlabel("Horas de frío acumuladas", fontsize=10)
ax1.set_ylabel("Densidad de probabilidad", fontsize=10)
ax1.legend(fontsize=8.5, framealpha=0.9)
ax1.grid(axis="y", linestyle=":", alpha=0.4, color="#AAAAAA")

# Panel derecho — Q-Q plot contra la distribución seleccionada
res_mejor = resultados_ajuste[mejor_nombre]
(osm, osr), (slope_qq, intercept_qq, r_qq) = stats.probplot(
    x, dist=res_mejor["dist"], sparams=res_mejor["params"]
)
ax2.scatter(osm, osr, color="#2C5F8A", s=45, zorder=5, label="Datos observados")
ax2.plot(osm, slope_qq * np.array(osm) + intercept_qq,
         color="#404040", linewidth=1.8, linestyle="--", label="Línea de referencia")
ax2.set_title(f"Gráfico Q-Q — distribución {mejor_nombre}\nHoras de frío (1990–2024)",
              fontsize=12, fontweight="bold", loc="center")
ax2.set_xlabel("Cuantiles teóricos", fontsize=10)
ax2.set_ylabel("Cuantiles observados", fontsize=10)
ax2.legend(fontsize=9, framealpha=0.9)
ax2.grid(linestyle=":", alpha=0.4, color="#AAAAAA")

plt.tight_layout()
plt.figtext(0.99, 0.01, "Fuente: Elaboración propia a partir de datos ERA5-Land (Open-Meteo).",
            ha="right", fontsize=8, color="#555555")
plt.show()

# ── Resumen numérico ──────────────────────────────────────────────────────
print("─" * 56)
print("TEST DE SHAPIRO-WILK (normalidad)")
print("─" * 56)
print(f"  Estadístico W : {sw_stat:.4f}")
print(f"  p-valor       : {sw_p:.4f}")
print(f"  Conclusión    : {'No se rechaza H₀ (distribución compatible con Normal)' if sw_p > 0.05 else 'Se rechaza H₀ (distribución no Normal)'}")
print()
print("─" * 56)
print("COMPARACIÓN DE DISTRIBUCIONES CANDIDATAS (AIC)")
print("─" * 56)
for nombre, res in sorted(resultados_ajuste.items(), key=lambda x: x[1]["aic"]):
    marca = " ◄ seleccionada" if nombre == mejor_nombre else ""
    print(f"  {nombre:<12}  AIC={res['aic']:7.2f}   KS p={res['ks_p']:.3f}{marca}")
print(f"\nDistribución seleccionada para Monte Carlo: {mejor_nombre}")


## Síntesis e interpretación agronómica

A partir del análisis de las series históricas 1990–2024, se presentan las principales conclusiones en relación con la aptitud agroclimática del sitio para el cultivo de pistacho.

**Régimen térmico estival.** Los valores de temperatura máxima media en enero–febrero se sitúan en el rango óptimo para el cultivo (35–38 °C) durante la mayor parte del período analizado. La tendencia al calentamiento implica que, de mantenerse, el sitio se aproximará al límite superior de ese rango y aumentará la frecuencia de días con temperaturas superiores a los 38 y 40 °C. Estas condiciones no inviabilizan el cultivo, pero incrementan el riesgo de estrés térmico durante la maduración y pueden reducir el calibre y el porcentaje de apertura del fruto.

**Período de brotación y floración.** El incremento en los Grados-Día de Crecimiento durante agosto–octubre indica que la acumulación de calor primaveral es creciente. Esto puede adelantar el inicio de la brotación y ampliar la asincronía entre la floración masculina y femenina, aspecto de relevancia especial en una especie dioica como el pistacho. Un mayor desajuste fenológico entre polinizadores y pistiladas podría reducir la eficiencia de la polinización y, en consecuencia, el cuajado de frutos.

**Acumulación de frío invernal.** La acumulación de horas de frío se mantiene por encima del umbral crítico de 800 horas en la mayoría de los años del período, lo que indica que el sitio dispone, en términos generales, del frío invernal requerido para la dormición del cultivo. Se destaca que el análisis se basa en el Modelo de Horas de Frío Simple (umbral < 7 °C), que es el más conservador de los disponibles. La aplicación del Modelo de Utah, que pondera las horas según el rango térmico, podría modificar estas estimaciones y constituye una mejora metodológica recomendada para estudios futuros.

**Demanda hídrica y aridez.** El déficit hídrico anual es consistentemente elevado, lo que confirma que el riego es indispensable para la producción en este sitio. El índice de aridez de De Martonne ratifica la clasificación del sitio como árido a hiperárido, característica del piedemonte mendocino no irrigado. Una tendencia creciente en el déficit implicaría una mayor demanda de riego que deberá considerarse en la planificación de la infraestructura hídrica y en la evaluación económica del proyecto.

**Limitaciones del análisis.** Los datos provienen del reanálisis ERA5-Land y no han sido validados contra estaciones meteorológicas locales, lo que introduce una incertidumbre que debe tenerse en cuenta al interpretar los resultados. La serie de 35 años es adecuada para el análisis de tendencias, pero limita la potencia estadística de los tests aplicados, particularmente en variables con alta variabilidad interanual. Estos aspectos deberán considerarse al extrapolar las conclusiones a horizontes de planificación de largo plazo.

## Parámetros calibrados — exportar al Módulo 1

Se consolidan los indicadores climáticos derivados del análisis histórico en un diccionario de parámetros (`PARAMS_CLIMA_JOCOLI`) que alimenta la simulación de Monte Carlo del módulo siguiente. La distribución de probabilidad de las horas de frío se asigna automáticamente en función del resultado del análisis AIC realizado en la sección anterior, en lugar de asumir normalidad a priori.

In [ ]:
# Mapeo nombre legible → identificador scipy
_nombre_a_id = {"Normal": "norm", "Log-normal": "lognorm", "Gamma": "gamma"}
_dist_id     = _nombre_a_id[mejor_nombre]
_dist_params = resultados_ajuste[mejor_nombre]["params"]

PARAMS_CLIMA_JOCOLI = {
    "horas_frio_media":          float(horas_frio.mean()),
    "horas_frio_std":            float(horas_frio.std()),
    "horas_frio_dist":           _dist_id,
    "horas_frio_dist_params":    _dist_params,
    "prob_anio_critico":         float((horas_frio < UMBRAL_FRIO_CRITICO).mean()),
    "prob_helada_tardia":        float((heladas > 0).mean()),
    "calor_verano_media":        float(calor_verano.mean()),
    "deficit_hidrico_media_mm":  float(deficit_hidrico.mean()),
}

print("PARAMS_CLIMA_JOCOLI = {")
for k, v in PARAMS_CLIMA_JOCOLI.items():
    print(f"    {k!r}: {v},")
print("}")
